# Signals of Prestige in Film Discourse: A Transformer-Based Approach to Predicting Best Picture Winners

### By: Ed Hou, Si Qin Huang, Alejandro Mendez

---

In [3]:
# pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 9.1 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertModel, AutoModel, AutoTokenizer
from typing import Optional

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import scipy.sparse as sp
import numpy as np

## Dataset (NEEDS TO BE IMPLEMENTED)

Load web-scraped reviews from Twitter and Metacritic

## Pre-processing (NEEDS TO BE IMPLEMENTED)
This step should do the follow:
  - Filter by date window
  - Remove noise and spam
  - Volume normalization
  - Text cleaning

## Model

**Pipeline**: <br>
BERT  → [CLS] per review  → WeightedAggregator → review film vector <br>
BERTweet → [CLS] per tweet → WeightedAggregator → tweet film vector <br>
Concatenate → Classification Head → P(win)

**Weighted Aggregator**: aggregates review embeddings from BERT and BERTweet into a single film vector

### BERT + BERTweet

In [4]:
class OscarPredictor(nn.Module):
    def __init__(
        self,
        review_model_name="bert-base-uncased",
        tweet_model_name="vinai/bertweet-base",
        hidden_dim=768,
        dropout=0.3
    ):
        super(OscarPredictor, self).__init__()

        # Transformer encoders
        self.review_encoder = AutoModel.from_pretrained(review_model_name)
        self.tweet_encoder = AutoModel.from_pretrained(tweet_model_name)

        self.review_scorer = nn.Linear(hidden_dim, 1, bias=False)
        self.tweet_scorer = nn.Linear(hidden_dim, 1, bias=False)

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(
        self,
        review_input_ids,
        review_attention_mask,
        tweet_input_ids,
        tweet_attention_mask
    ):
        """
        Inputs shape:
        - review_input_ids: (batch_size, num_reviews, seq_len)
        - tweet_input_ids: (batch_size, num_tweets, seq_len)
        """

        batch_size, num_reviews, review_seq_len = review_input_ids.shape
        _, num_tweets, tweet_seq_len = tweet_input_ids.shape

        # Process reviews
        review_input_ids = review_input_ids.reshape(-1, review_seq_len)
        review_attention_mask = review_attention_mask.reshape(-1, review_seq_len)

        review_outputs = self.review_encoder(
            input_ids=review_input_ids,
            attention_mask=review_attention_mask
        )

        # CLS token representation
        review_cls = review_outputs.last_hidden_state[:, 0, :]
        review_cls = review_cls.reshape(batch_size, num_reviews, -1)

        # Weighted aggregation
        review_weights = F.softmax(self.review_scorer(review_cls), dim=1)
        review_agg = (review_weights * review_cls).sum(dim=1)

        # Process tweets
        tweet_input_ids = tweet_input_ids.reshape(-1, tweet_seq_len)
        tweet_attention_mask = tweet_attention_mask.reshape(-1, tweet_seq_len)

        tweet_outputs = self.tweet_encoder(
            input_ids=tweet_input_ids,
            attention_mask=tweet_attention_mask
        )

        tweet_cls = tweet_outputs.last_hidden_state[:, 0, :]
        tweet_cls = tweet_cls.reshape(batch_size, num_tweets, -1)

        tweet_weights = F.softmax(self.tweet_scorer(tweet_cls), dim=1)
        tweet_agg = (tweet_weights * tweet_cls).sum(dim=1)

        # Combine reviews and tweets
        film_vector = torch.cat([review_agg, tweet_agg], dim=1)

        # Classification
        output = self.classifier(film_vector)

        return output

In [5]:
# ---------------------------------------------------------------------------
# Example usage
# ---------------------------------------------------------------------------
BERT_TOKENIZER     = "bert-base-uncased"
BERTWEET_TOKENIZER = "vinai/bertweet-base"
NUM_REVIEWS        = 5
NUM_TWEETS         = 10
REVIEW_SEQ_LEN     = 512
TWEET_SEQ_LEN      = 128   # BERTweet max is 128
BATCH_SIZE         = 4     # four films

review_tokenizer = AutoTokenizer.from_pretrained(BERT_TOKENIZER)
tweet_tokenizer  = AutoTokenizer.from_pretrained(BERTWEET_TOKENIZER)

model     = OscarPredictor()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.BCELoss()

# Tokenization
sample_reviews = ["An incredible cinematic achievement."] * NUM_REVIEWS
sample_tweets  = ["Best movie of the year!"] * NUM_TWEETS

def tokenize(tokenizer, texts, seq_len):
    encoded = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=seq_len,
        return_tensors="pt"
    )
    return encoded["input_ids"], encoded["attention_mask"]

review_ids,  review_mask = tokenize(review_tokenizer, sample_reviews, REVIEW_SEQ_LEN)
tweet_ids,   tweet_mask  = tokenize(tweet_tokenizer,  sample_tweets,  TWEET_SEQ_LEN)

# Stack to (batch_size, num_texts, seq_len)
review_ids  = review_ids.unsqueeze(0).expand(BATCH_SIZE, -1, -1).reshape(BATCH_SIZE, NUM_REVIEWS, REVIEW_SEQ_LEN)
review_mask = review_mask.unsqueeze(0).expand(BATCH_SIZE, -1, -1).reshape(BATCH_SIZE, NUM_REVIEWS, REVIEW_SEQ_LEN)
tweet_ids   = tweet_ids.unsqueeze(0).expand(BATCH_SIZE, -1, -1).reshape(BATCH_SIZE, NUM_TWEETS,  TWEET_SEQ_LEN)
tweet_mask  = tweet_mask.unsqueeze(0).expand(BATCH_SIZE, -1, -1).reshape(BATCH_SIZE, NUM_TWEETS,  TWEET_SEQ_LEN)

# Training step
labels = torch.tensor([[1.], [0.], [0.], [0.]])  # one winner per year

model.train()
optimizer.zero_grad()
probs = model(review_ids, review_mask, tweet_ids, tweet_mask)
loss  = criterion(probs, labels)
loss.backward()
optimizer.step()
print(f"Training loss: {loss.item():.4f}")

# Inference
model.eval()
with torch.no_grad():
    probs = model(review_ids, review_mask, tweet_ids, tweet_mask)

print(f"Output shape : {probs.shape}")
for i, p in enumerate(probs):
    print(f"  Film {i} P(win): {p.item():.4f}")
winner = probs.squeeze(-1).argmax().item()
print(f"Predicted winner: film {winner}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/bertweet-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Training loss: 0.7046
Output shape : torch.Size([4, 1])
  Film 0 P(win): 0.4883
  Film 1 P(win): 0.4883
  Film 2 P(win): 0.4883
  Film 3 P(win): 0.4883
Predicted winner: film 0


### BERT (On reviews + tweets)

In [16]:
class OscarPredictorBERT(nn.Module):
    """
    Baseline model: single shared BERT encoder for both reviews and tweets.
    Uses mean pooling instead of learned aggregation.
    """
    def __init__(
        self,
        model_name="bert-base-uncased",
        hidden_dim=768,
        dropout=0.3
    ):
        super(OscarPredictorBERT, self).__init__()

        # Single shared encoder for both reviews and tweets
        self.encoder = AutoModel.from_pretrained(model_name)

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def encode(self, input_ids, attention_mask):
        """
        Encode a batch of texts and return mean-pooled CLS representations.

        input_ids     : (batch_size, num_texts, seq_len)
        attention_mask: (batch_size, num_texts, seq_len)
        returns       : (batch_size, hidden_dim)
        """
        batch_size, num_texts, seq_len = input_ids.shape

        input_ids = input_ids.reshape(-1, seq_len)
        attention_mask = attention_mask.reshape(-1, seq_len)

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls = outputs.last_hidden_state[:, 0, :]
        cls = cls.view(batch_size, num_texts, -1)
        return cls.mean(dim=1)

    def forward(
        self,
        review_input_ids,
        review_attention_mask,
        tweet_input_ids,
        tweet_attention_mask
    ):
        review_agg = self.encode(review_input_ids, review_attention_mask)
        tweet_agg  = self.encode(tweet_input_ids,  tweet_attention_mask)

        film_vector = torch.cat([review_agg, tweet_agg], dim=1)

        return self.classifier(film_vector)

In [17]:
# ---------------------------------------------------------------------------
# Example usage
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    from transformers import AutoTokenizer

    TOKENIZER   = "bert-base-uncased"
    NUM_REVIEWS = 5
    NUM_TWEETS  = 10
    SEQ_LEN     = 128
    BATCH_SIZE  = 4   # four films

    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER)
    model     = OscarPredictorBERT()
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
    criterion = nn.BCELoss()

    # Tokenization
    sample_reviews = ["An incredible cinematic achievement."] * NUM_REVIEWS
    sample_tweets  = ["Best movie of the year!"] * NUM_TWEETS

    def tokenize(texts, seq_len):
        """Tokenize a list of strings, return input_ids and attention_mask."""
        encoded = tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=seq_len,
            return_tensors="pt"
        )
        return encoded["input_ids"], encoded["attention_mask"]

    # Tokenize once, then repeat across the batch of films
    review_ids,  review_mask  = tokenize(sample_reviews, SEQ_LEN)
    tweet_ids,   tweet_mask   = tokenize(sample_tweets,  SEQ_LEN)

    # Stack to (batch_size, num_texts, seq_len)
    review_ids   = review_ids.unsqueeze(0).expand(BATCH_SIZE, -1, -1)
    review_mask  = review_mask.unsqueeze(0).expand(BATCH_SIZE, -1, -1)
    tweet_ids    = tweet_ids.unsqueeze(0).expand(BATCH_SIZE, -1, -1)
    tweet_mask   = tweet_mask.unsqueeze(0).expand(BATCH_SIZE, -1, -1)

    # Training step
    labels = torch.tensor([[1.], [0.], [0.], [0.]])  # one winner per year

    model.train()
    optimizer.zero_grad()
    probs = model(review_ids, review_mask, tweet_ids, tweet_mask)
    loss  = criterion(probs, labels)
    loss.backward()
    optimizer.step()
    print(f"Training loss: {loss.item():.4f}")

    # Inference
    model.eval()
    with torch.no_grad():
        probs = model(review_ids, review_mask, tweet_ids, tweet_mask)

    print(f"Output shape : {probs.shape}")
    for i, p in enumerate(probs):
        print(f"  Film {i} P(win): {p.item():.4f}")
    winner = probs.squeeze(-1).argmax().item()
    print(f"Predicted winner: film {winner}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Training loss: 0.6709
Output shape : torch.Size([4, 1])
  Film 0 P(win): 0.4314
  Film 1 P(win): 0.4314
  Film 2 P(win): 0.4314
  Film 3 P(win): 0.4314
Predicted winner: film 0


### TF-IDF + Logistic Regression

In [8]:
class OscarPredictorTFIDF:
    """
    Baseline model: TF-IDF + Logistic Regression.
    Reviews and tweets are each vectorized separately then concatenated.
    """

    def __init__(self, max_features=10000, C=1.0):
        # Separate vectorizers so each modality gets its own vocab
        self.review_vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=(1, 2),    # unigrams + bigrams
            sublinear_tf=True,     # apply log normalization to term freq
            strip_accents="unicode",
            min_df=2,              # ignore terms appearing in fewer than 2 films
        )
        self.tweet_vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=(1, 2),
            sublinear_tf=True,
            strip_accents="unicode",
            min_df=2,
        )
        self.classifier = LogisticRegression(
            C=C,                   # inverse regularization strength
            max_iter=1000,
            class_weight="balanced",  # handles 1 winner vs many nominees imbalance
        )

    def _prepare_features(self, reviews, tweets, fit=False):
        """
        Vectorize reviews and tweets and concatenate into one feature matrix.

        reviews : list of str — one string per film (all reviews concatenated)
        tweets  : list of str — one string per film (all tweets concatenated)
        returns : sparse matrix (num_films, 2 * max_features)
        """
        if fit:
            review_features = self.review_vectorizer.fit_transform(reviews)
            tweet_features  = self.tweet_vectorizer.fit_transform(tweets)
        else:
            review_features = self.review_vectorizer.transform(reviews)
            tweet_features  = self.tweet_vectorizer.transform(tweets)

        # Horizontally stack review and tweet features
        return sp.hstack([review_features, tweet_features])

    def fit(self, reviews, tweets, labels):
        """
        Train the model.

        reviews : list of str, one per film
        tweets  : list of str, one per film
        labels  : list of int, 1 = winner, 0 = nominee
        """
        X = self._prepare_features(reviews, tweets, fit=True)
        self.classifier.fit(X, labels)

    def predict_proba(self, reviews, tweets):
        """
        Returns P(win) for each film.

        returns: np.array of shape (num_films,)
        """
        X = self._prepare_features(reviews, tweets, fit=False)
        return self.classifier.predict_proba(X)[:, 1]

    def predict_winner(self, reviews, tweets):
        """
        Returns the index of the predicted winner within the provided films
        (intended to be called with one year's nominees at a time).
        """
        probs = self.predict_proba(reviews, tweets)
        return np.argmax(probs)

In [9]:
# ---------------------------------------------------------------------------
# Example usage
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    # Each entry is one film — concatenate all its reviews into one string
    # and all its tweets into one string before passing in
    train_reviews = [
        "a masterpiece of cinema stunning visuals",
        "disappointing formulaic and dull",
        "emotional powerful and beautifully acted",
    ]
    train_tweets = [
        "best movie of the year must see",
        "total waste of time avoid",
        "cried three times incredible film",
    ]
    train_labels = [1, 0, 0]

    model = OscarPredictorTFIDF()
    model.fit(train_reviews, train_tweets, train_labels)

    test_reviews = ["visually stunning and deeply moving"]
    test_tweets  = ["everyone is talking about this film"]
    print(f"P(win): {model.predict_proba(test_reviews, test_tweets)[0]:.4f}")

P(win): 0.3736


## Evaluation (NEEDS TO BE IMPLEMENTED)
Methods:
  - Accuracy
  - Top-1, Top-5
  - Others?